# Trading Strategy Based on Z-Score Thresholds

Let z_i(t) be the z-score of stock i at time t.

Trading Rules:
- If z_i(t) > 3: Take a short position in stock i
  - Indicates stock is significantly overvalued (3 standard deviations above mean)
- If z_i(t) < -3: Take a long position in stock i 
  - Indicates stock is significantly undervalued (3 standard deviations below mean)
- Otherwise (-3 ≤ z_i(t) ≤ 3): Hold current position
  - Stock price is within normal range

This strategy aims to profit from mean reversion by:
1. Selling overvalued stocks (expecting price decrease)
2. Buying undervalued stocks (expecting price increase)

We will evaluate this strategy through backtesting to measure historical performance.


In [51]:
import pandas as pd
z_scores = pd.read_csv('/Users/dhairya/cs projects/DS303 final project/dataset/z_scores.csv')
cluster_and_price_data = pd.read_csv('/Users/dhairya/cs projects/DS303 final project/dataset/cluster_price_data.csv')

In [52]:
for col in z_scores.columns:
    if col != 'Date':
        z_scores[col] = z_scores[col].apply(lambda x: -1 if x > 3 else (1 if x < -3 else 0))
z_scores

,Date,20MICRONS_closing_0,ADANIPOWER_closing_0,ALKALI_closing_0,APCL_closing_0,APTECHT_closing_0,ARIES_closing_0,AVADHSUGAR_closing_0,AVTNPL_closing_0,BAJAJHIND_closing_0,...,VOLTAMP_closing_4,VSSL_closing_4,VTL_closing_4,WABAG_closing_4,WELCORP_closing_4,WONDERLA_closing_4,WSTCSTPAPR_closing_4,YAARI_closing_4,ZODIAC_closing_4,ZODIACLOTH_closing_4
0,2020-01-02,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2020-01-06,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2020-01-07,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2020-01-08,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2020-01-09,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,2021-12-27,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
489,2021-12-28,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
490,2021-12-29,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
491,2021-12-30,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [53]:
num_ones = (z_scores.iloc[:, 1:] == 1).sum().sum()
num_neg_ones = (z_scores.iloc[:, 1:] == -1).sum().sum()

print(f"Number of 1s (long positions): {num_ones}")
print(f"Number of -1s (short positions): {num_neg_ones}")

Number of 1s (long positions): 1289
Number of -1s (short positions): 6704


In [54]:
cols = [col for col in z_scores.columns if col != 'Date']

cumulative_sums = pd.concat([z_scores[col].cumsum() for col in cols], axis=1)
cumulative_sums.columns = cols
cumulative_sums

,20MICRONS_closing_0,ADANIPOWER_closing_0,ALKALI_closing_0,APCL_closing_0,APTECHT_closing_0,ARIES_closing_0,AVADHSUGAR_closing_0,AVTNPL_closing_0,BAJAJHIND_closing_0,BALPHARMA_closing_0,...,VOLTAMP_closing_4,VSSL_closing_4,VTL_closing_4,WABAG_closing_4,WELCORP_closing_4,WONDERLA_closing_4,WSTCSTPAPR_closing_4,YAARI_closing_4,ZODIAC_closing_4,ZODIACLOTH_closing_4
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,0,-4,-4,0,0,0,0,0,-5,-4,...,0,0,0,0,0,0,0,-26,-30,-30
489,0,-4,-4,0,0,0,0,0,-5,-4,...,0,0,0,0,0,0,0,-26,-30,-30
490,0,-4,-4,0,0,0,0,0,-5,-4,...,0,0,0,0,0,0,0,-26,-30,-30
491,0,-4,-4,0,0,0,0,0,-5,-4,...,0,0,0,0,0,0,0,-26,-30,-30


In [38]:
cumulative_sums.index = z_scores['Date']
price_data = pd.read_csv('/Users/dhairya/cs projects/DS303 final project/dataset/cluster_price_data.csv')
price_data.set_index('Date', inplace=True)

position_values = cumulative_sums.multiply(price_data)
position_values


,20MICRONS_closing_0,3MINDIA_closing_2,5PAISA_closing_3,63MOONS_closing_1,A2ZINFRA_closing_1,AARTIDRUGS_closing_3,AARVEEDEN_closing_1,AAVAS_closing_2,ABBOTINDIA_closing_3,ABB_closing_2,...,ZEEL_closing_2,ZEEMEDIA_closing_1,ZENSARTECH_closing_3,ZENTEC_closing_1,ZODIACLOTH_closing_4,ZODIAC_closing_4,ZOTA_closing_3,ZUARIGLOB_closing_0,ZUARI_closing_0,ZYDUSWELL_closing_3
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-06,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-07,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-08,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-09,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,0.0,-199707.2,0.0,0.0,0.0,-1569.00,0.0,0.0,-488650.5,0.0,...,1271.8,0.0,0.0,0.0,-3178.5,-3178.5,0.0,0.0,0.0,0.0
2021-12-28,0.0,-199755.2,0.0,0.0,0.0,-1557.45,0.0,0.0,-495316.9,0.0,...,1286.0,0.0,0.0,0.0,-3234.0,-3234.0,0.0,0.0,0.0,0.0
2021-12-29,0.0,-199600.4,0.0,0.0,0.0,-1586.25,0.0,0.0,-502232.9,0.0,...,1271.8,0.0,0.0,0.0,-3210.0,-3210.0,0.0,0.0,0.0,0.0


In [55]:
z_scores

,Date,20MICRONS_closing_0,ADANIPOWER_closing_0,ALKALI_closing_0,APCL_closing_0,APTECHT_closing_0,ARIES_closing_0,AVADHSUGAR_closing_0,AVTNPL_closing_0,BAJAJHIND_closing_0,...,VOLTAMP_closing_4,VSSL_closing_4,VTL_closing_4,WABAG_closing_4,WELCORP_closing_4,WONDERLA_closing_4,WSTCSTPAPR_closing_4,YAARI_closing_4,ZODIAC_closing_4,ZODIACLOTH_closing_4
0,2020-01-02,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2020-01-06,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2020-01-07,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2020-01-08,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2020-01-09,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,2021-12-27,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
489,2021-12-28,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
490,2021-12-29,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
491,2021-12-30,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [56]:
cluster_and_price_data

,Date,20MICRONS_closing_0,3MINDIA_closing_2,5PAISA_closing_3,63MOONS_closing_1,A2ZINFRA_closing_1,AARTIDRUGS_closing_3,AARVEEDEN_closing_1,AAVAS_closing_2,ABB_closing_2,...,ZEELEARN_closing_1,ZEEMEDIA_closing_1,ZENSARTECH_closing_3,ZENTEC_closing_1,ZODIAC_closing_4,ZODIACLOTH_closing_4,ZOTA_closing_3,ZUARI_closing_0,ZUARIGLOB_closing_0,ZYDUSWELL_closing_3
0,2020-01-02,35.55,21381.15,211.30,110.30,8.20,147.55,14.30,1987.90,1294.20,...,19.15,6.05,180.30,57.25,172.25,172.25,189.90,58.20,58.20,1473.80
1,2020-01-06,32.80,21089.70,198.05,104.20,8.05,140.54,14.15,1990.20,1287.00,...,19.45,5.90,181.95,57.85,167.20,167.20,185.85,61.30,61.30,1438.05
2,2020-01-07,33.20,21660.40,204.75,107.75,8.20,143.28,13.95,1970.80,1306.75,...,20.00,6.00,184.00,58.90,171.20,171.20,186.40,73.55,73.55,1428.50
3,2020-01-08,32.70,21238.05,195.95,105.55,8.00,141.28,13.95,1938.55,1310.70,...,19.60,6.00,183.90,58.20,169.75,169.75,185.50,72.50,72.50,1446.70
4,2020-01-09,33.40,21272.20,195.50,107.80,7.95,144.81,14.10,1978.35,1322.10,...,19.35,5.95,184.85,60.45,171.90,171.90,185.35,68.95,68.95,1454.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,2021-12-27,57.10,24963.40,407.60,207.15,7.90,523.00,29.55,2488.45,2195.50,...,13.90,12.55,470.10,215.30,105.95,105.95,395.95,134.70,134.70,1859.85
489,2021-12-28,59.75,24969.40,407.70,217.50,8.25,519.15,31.00,2490.30,2207.10,...,14.05,13.15,498.70,216.75,107.80,107.80,398.10,140.10,140.10,1920.70
490,2021-12-29,59.60,24950.05,405.50,228.35,8.65,528.75,29.45,2551.05,2226.60,...,13.95,12.95,524.60,215.40,107.00,107.00,395.65,142.60,142.60,1903.40
491,2021-12-30,62.35,25047.00,393.40,217.30,9.05,522.70,28.35,2575.75,2235.40,...,14.70,12.80,518.70,216.70,108.00,108.00,394.35,140.85,140.85,1901.25


In [58]:
for col in z_scores.columns:
    if col != 'Date':
        z_scores[col] = z_scores[col]*cluster_and_price_data[col]

In [63]:

abs_sum = z_scores.iloc[:, 1:].abs().sum().sum()
abs_sum

np.float64(5621207.392)

In [66]:
z_scores = z_scores.cumsum()

In [104]:
profit = position_values.copy()

In [105]:
profit

,20MICRONS_closing_0,3MINDIA_closing_2,5PAISA_closing_3,63MOONS_closing_1,A2ZINFRA_closing_1,AARTIDRUGS_closing_3,AARVEEDEN_closing_1,AAVAS_closing_2,ABBOTINDIA_closing_3,ABB_closing_2,...,ZEEL_closing_2,ZEEMEDIA_closing_1,ZENSARTECH_closing_3,ZENTEC_closing_1,ZODIACLOTH_closing_4,ZODIAC_closing_4,ZOTA_closing_3,ZUARIGLOB_closing_0,ZUARI_closing_0,ZYDUSWELL_closing_3
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-06,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-07,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-08,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2020-01-09,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,0.0,-199707.2,0.0,0.0,0.0,-1569.00,0.0,0.0,-488650.5,0.0,...,1271.8,0.0,0.0,0.0,-3178.5,-3178.5,0.0,0.0,0.0,0.0
2021-12-28,0.0,-199755.2,0.0,0.0,0.0,-1557.45,0.0,0.0,-495316.9,0.0,...,1286.0,0.0,0.0,0.0,-3234.0,-3234.0,0.0,0.0,0.0,0.0
2021-12-29,0.0,-199600.4,0.0,0.0,0.0,-1586.25,0.0,0.0,-502232.9,0.0,...,1271.8,0.0,0.0,0.0,-3210.0,-3210.0,0.0,0.0,0.0,0.0


In [110]:
from tqdm import tqdm

for col in tqdm(profit.columns):
    if col != 'Date':
        for i in range(len(profit)):
            profit.iloc[i][col] = profit.iloc[i][col] - z_scores.iloc[i][col]


  0%|          | 0/1262 [00:00<?, ?it/s]/var/folders/wl/q8prsl_x2_g_hl0vv6cf0xj00000gn/T/ipykernel_34153/2088378583.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  profit.iloc[i][col] = profit.iloc[i][col] - z_scores.iloc[i][col]
100%|██

In [119]:
profit.to_csv('/Users/dhairya/cs projects/DS303 final project/dataset/profits.csv')